# Embedding Models 选择笔记

本笔记记录考古 RAG 项目中可选的 embedding 模型及其适用场景。

核心问题：把中文考古文本变成向量后，检索时要找得到、找得准。

## 1. 快速决策表

| 场景 | 推荐模型 | 维度 | 说明 |
|---|---|---|---|
| 有 OpenAI key，想最快跑通 | `text-embedding-3-small` | 1536 | 便宜、快、中英文都不错 |
| 要中文考古术语召回好，无需本地 GPU | `BAAI/bge-m3` via SiliconFlow | 1024 | 在线 API、 cheaper、中文 RAG 首选 |
| 要质量最好，预算够 | `text-embedding-3-large` | 3072 | OpenAI 最强 embedding |
| 要本地部署，中文为主 | `BAAI/bge-m3` | 1024 | 中文 RAG 首选，支持混合检索 |
| 要本地多语言 | `intfloat/multilingual-e5-large` | 1024 | 多语言任务稳 |
| 完全离线，轻量 | `nomic-embed-text` (Ollama) | 768 | 本地 Ollama 即可跑 |

## 2. OpenAI 模型

### `text-embedding-3-small`
- 维度：1536
- 优点：成本低、速度快、对中英文都足够好
- 缺点：对极专业术语不如 bge-m3
- 适用：项目初期、快速验证、预算敏感

### `text-embedding-3-large`
- 维度：3072
- 优点：OpenAI 目前能力最强的 embedding
- 缺点：价格约为 small 的 6-7 倍
- 适用：对检索质量要求极高、数据量不大

### `text-embedding-ada-002`
- 旧版，不推荐新项目使用。3-small 更快更便宜。

## 3. 中文/本地模型

### `BAAI/bge-m3`（最推荐）
- 维度：1024
- 特点：
  - 同时输出 dense、sparse、multi-vector 三种表示
  - 对中文考古术语（如“竖穴土坑墓”“陶鬲豆罐”）召回率高
  - 配合 Milvus 混合检索（dense + BM25）效果最佳
- 使用：`sentence-transformers` 或 `FlagEmbedding` 库

### `BAAI/bge-large-zh-v1.5`
- 维度：1024
- 特点：纯 dense，中文语义理解强
- 缺点：没有 sparse，术语匹配不如 bge-m3

### `intfloat/multilingual-e5-large`
- 维度：1024
- 特点：多语言能力强，适合中英混合语料
- 缺点：中文专门任务略弱于 bge 系列

## 4. 完全本地轻量方案

### Ollama 嵌入模型
| 模型 | 维度 | 说明 |
|---|---|---|
| `nomic-embed-text` | 768 | 体积小，速度快 |
| `mxbai-embed-large` | 1024 | 质量较好 |
| `snowflake-arctic-embed` | 1024 | 检索优化 |

用法：
```bash
ollama pull nomic-embed-text
ollama serve
```

base URL 填 `http://localhost:11434/v1`，模型名填 `nomic-embed-text`。

## 5. 本项目建议

**阶段 1（验证）**：`BAAI/bge-m3` via SiliconFlow
- endpoint: `https://api.siliconflow.cn/v1`，获取 key 见 https://cloud.siliconflow.cn/account/ak
- 中文考古术语召回优于 OpenAI text-embedding-3-small
- `scripts/build_milvus_index.py` 已支持 OpenAI 兼容接口，直接填 SiliconFlow 参数即可
- 运行：`cp .env.example .env && source .env && uv run python scripts/build_milvus_index.py`（编辑 .env 填入 API key）
- 或直接：`OPENAI_BASE_URL=https://api.siliconflow.cn/v1 OPENAI_API_KEY=sk-... EMBEDDING_MODEL=BAAI/bge-m3 uv run python scripts/build_milvus_index.py`

**阶段 2（优化）**：本地 `BAAI/bge-m3`
- 本地部署后替换模型名即可
- 开启 Milvus 混合检索，提升专业术语召回

**阶段 3（生产）**：按需选择
- 小规模、高质量：OpenAI `text-embedding-3-large` 或 SiliconFlow `Pro/BAAI/bge-m3`
- 大规模、低成本：本地 bge-m3 + Milvus Standalone

## 6. 测试检索质量的简单方法

1. 建索引
2. 用几个真实问题测试：`M12 是什么年代`、`竖穴土坑墓 典型随葬品`、`湖南楚汉漆器 分期`
3. 看 top-3 结果是否包含相关报告原文
4. 如果专业术语搜不到，换 bge-m3 或加 BM25